In [ ]:
cd ..

In [ ]:
cd ..

In [ ]:
cd ..

In [ ]:
from src.quality_metrics.evaluate_noise_resistance import load_examples, jaccard, find_all_common_questions
from scipy.stats import kendalltau as scipy_kendalltau
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import json

### Noise Resistance

In [ ]:
from dataclasses import dataclass

def load_examples(folder: Path, check: bool = False):
    examples = {}
    for path in folder.rglob("*.json"):
        with open(path) as f:
            data = json.load(f)
        
        question = data.get("question", "")
        if question:
            if check:
                if data.get("found") == True:
                    examples[question] = data   
            else: 
                examples[question] = data

    return examples

def load_noise_examples(folder: Path, check: bool = False):
    examples = {}
    for path in folder.rglob("*.json"):
        with open(path) as f:
            data = json.load(f)
        
        question = data.get("question", "")
        if question:
            if check:
                if data.get("noise_robust") == True and data.get("found") == True:
                    examples[question] = data   
            else: 
                examples[question] = data

    return examples

@dataclass
class OpsEntities:
    nodes: set
    edges: set

    def overlaps(self, other: "OpsEntities") -> bool:
        return bool(self.nodes & other.nodes) or bool(self.edges & other.edges)

def parse_ops(operations) -> list[tuple]:
    """Parse raw operations into normalized (op_type, entity) tuples."""
    parsed = []
    for op in operations:
        op_type = op[0]
        if op_type in ("delete_edge", "add_edge"):
            parsed.append((op_type, (op[1][0], op[1][1])))
        elif op_type in ("delete_node", "add_node"):
            parsed.append((op_type, op[1]))
    return parsed


def parse_noise(operations) -> OpsEntities:
    """Extract only the entities touched by noise ops (ignoring op type)."""
    nodes, edges = set(), set()
    for op in operations:
        op_type = op[0]
        if op_type in ("add_node", "delete_node"):
            nodes.add(op[1])
        elif op_type in ("add_edge", "delete_edge"):
            edges.add((op[1][0], op[1][1]))
    return OpsEntities(nodes=nodes, edges=edges)



def kendall_tau_ops(old_ops: list[tuple], new_ops: list[tuple]) -> float | None:
    """
    Kendall τ on the shared (op_type, entity) pairs, ranked by position.
    Returns None when fewer than 2 shared items exist.
    """
    old_map = {item: i for i, item in enumerate(old_ops)}
    new_map = {item: i for i, item in enumerate(new_ops)}
    shared  = list(set(old_map) & set(new_map))
    if len(shared) < 2:
        return None
    tau, _ = scipy_kendalltau(
        [old_map[x] for x in shared],
        [new_map[x] for x in shared],
    )
    return float(tau)

def noise_sensitivity(old_explanation, new_explanation) -> dict | int:
    if not new_explanation["noise"]["noise_robust"]:
        return -1

    noise_entities = parse_noise(new_explanation["noise"]["ops"])
    new_ops = parse_ops(new_explanation["operations"])
    old_ops = parse_ops(old_explanation["operations"])

    noise_nodes_in_new = [op for op in new_ops if op[0] in ("add_node", "delete_node") and op[1] in noise_entities.nodes]
    noise_edges_in_new = [op for op in new_ops if op[0] in ("add_edge", "delete_edge") and op[1] in noise_entities.edges]

    return {
        "jaccard_similarity": jaccard(set(new_ops), set(old_ops)),
        "cost_var": new_explanation["cost"] - old_explanation["cost"],
        "operation_var": new_explanation["num_operations"] - old_explanation["num_operations"],
        "noise_nodes_in_new": noise_nodes_in_new,
        "noise_edges_in_new": noise_edges_in_new,
        "noise_resistant": not noise_nodes_in_new and not noise_edges_in_new,
        "kendall_tau": kendall_tau_ops(old_ops, new_ops),
    }

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────

datasets = ["synthetic", "hotpotqa", "musique", "2wiki"]
cases    = ["delete_ops_ft", "all_ops_ff"]
noise_percentages = [10, 20, 30, 50]

CASE_TITLES = {
    "delete_ops_ft": "T → F",
    "all_ops_ff":    "F → F",
}

STYLE = {
    ("synthetic", "delete_ops_ft"): dict(color="#2176AE", marker="o", linestyle="-",  label="Synthetic"),
    ("synthetic", "all_ops_ff"):    dict(color="#2176AE", marker="o", linestyle="-",  label="Synthetic"),
    ("hotpotqa",  "delete_ops_ft"): dict(color="#E84855", marker="s", linestyle="--", label="HotpotQA"),
    ("hotpotqa",  "all_ops_ff"):    dict(color="#E84855", marker="s", linestyle="--", label="HotpotQA"),
    ("musique",  "delete_ops_ft"): dict(color="#4853E8", marker="s", linestyle="--", label="MuSique"),
    ("musique",  "all_ops_ff"):    dict(color="#4853E8", marker="s", linestyle="--", label="MuSique"),
    ("2wiki",  "delete_ops_ft"): dict(color="#535575", marker="s", linestyle="--", label="2Wiki"),
    ("2wiki",  "all_ops_ff"):    dict(color="#535575", marker="s", linestyle="--", label="2Wiki"),
}

# ── Data collection ───────────────────────────────────────────────────────────

results = {}

for dataset in datasets:
    for case in cases:
        key = (dataset, case)
        results[key] = {}

        for noise_percentage in noise_percentages:
            original_filepath = Path(f"src/counterfactuals/results/{dataset}/{case}")
            noise_filepath    = Path(
                f"src/counterfactuals/robustness/{dataset}/noise_resistance/{case[-2:]}/"
                f"noise_level_{noise_percentage}"
            )

            original_examples = load_examples(original_filepath, check=True)
            noisy_examples    = load_noise_examples(noise_filepath)

            common_questions = sorted(set(original_examples) & set(noisy_examples))

            jaccard_similarity = []
            kendall_taus       = []
            robust_case_temp   = 0
            total_valid_cases  = 0

            for q in common_questions:
                old_explanation = original_examples[q]
                new_explanation = noisy_examples[q]

                if new_explanation["noise"] == {}:
                    continue

                noise_sens = noise_sensitivity(old_explanation, new_explanation)

                if noise_sens == -1:
                    continue

                total_valid_cases += 1
                jaccard_similarity.append(noise_sens["jaccard_similarity"])
                if noise_sens["kendall_tau"] is not None:
                    kendall_taus.append(noise_sens["kendall_tau"])

                if noise_sens["noise_resistant"]:
                    robust_case_temp += 1

            results[key][noise_percentage] = {
                "jaccard": np.round(np.mean(jaccard_similarity), 4) if jaccard_similarity else 0.0,
                "kendall": np.round(np.mean(kendall_taus), 4)       if kendall_taus       else 0.0,
                "robust":  np.round(robust_case_temp / len(common_questions), 4) if common_questions else 0.0,
            }

            print(
                f"[{dataset} / {case}] Level {noise_percentage}% → "
                f"Valid: {total_valid_cases}/{len(common_questions)}"
            )

# ── Shared axis helper ────────────────────────────────────────────────────────

def _style_ax(ax, ylabel):
    ax.set_xlabel("Noise Level (%)", fontsize=20, labelpad=10)
    ax.set_ylabel(ylabel, fontsize=20, labelpad=10)
    ax.set_xlim(5, 55)
    ax.set_ylim(0, 105)
    ax.set_xticks(noise_percentages)
    ax.tick_params(axis="x", labelsize=15)
    ax.tick_params(axis="y", labelsize=15)
    ax.yaxis.set_major_locator(MultipleLocator(10))
    ax.yaxis.set_minor_locator(MultipleLocator(2))
    ax.grid(axis="y", which="major", linestyle="--", alpha=0.5)
    ax.grid(axis="y", which="minor", linestyle=":",  alpha=0.3)
    ax.legend(loc="lower left", fontsize=15, framealpha=0.85)

# ── Figure 1 — Robustness (2 subplots, one per case) ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

for ax, case in zip(axes, cases):
    for dataset in datasets:
        style = STYLE[(dataset, case)]
        y_vals = [results[(dataset, case)][n]["robust"] * 100 for n in noise_percentages]
        ax.plot(noise_percentages, y_vals, **style)
    _style_ax(ax, "Robust Cases (%)" if ax is axes[0] else "")

plt.tight_layout()
# plt.savefig("src/counterfactuals/notebooks/noise_robustness.pdf", format="pdf")
plt.show()

plt.tight_layout()
# plt.savefig("src/counterfactuals/notebooks/noise_kendall_tau.pdf", format="pdf")
plt.show()
